In [1]:
# %%
import os, sys
import numpy as np
from utils import get_config
import numpy as np
import torch
from tqdm import tqdm
from torch.utils.data import TensorDataset, DataLoader
from torch.nn.parallel import DistributedDataParallel, DataParallel
from utils import get_sigma_time, get_sample_time, VESDE, get_config, get_filepath
from model import UNet3DModel
import matplotlib.pyplot as plt
from torch_ema import ExponentialMovingAverage
import logging
import os
import sys
from os.path import join
import argparse

In [2]:
from dataclasses import dataclass

@dataclass
class args:
    start = 1999
    end = 2000
    config = './configs/config_dm_1900_2.json'
    disable_tqdm = False

In [3]:
config =get_config(args.config)
DEVICE = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')

model_folder = join(config.model.workdir, config.model.cosmo_dir)
checkpoint_dir = join(model_folder, config.model.checkpoint_dir)
data_root = config.data.path

input_type = config.data.input_type
target_type = config.data.target_type

In [4]:
model = UNet3DModel(config)
#model = DataParallel(model)
model = model.to(DEVICE)

ema = ExponentialMovingAverage(model.parameters(), decay=config.model.ema_rate)

sde = VESDE(
    config.model.sigma_min, config.model.sigma_max, 
    config.model.num_scales, 
    config.model.T, config.model.sampling_eps
)

# Check for existing checkpoint
checkpoint_path = join(checkpoint_dir, 'checkpoint.pth')
if os.path.isfile(checkpoint_path):
    loaded_state = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(loaded_state['model'], strict=False)
    ema.load_state_dict(loaded_state['ema'])
    init_epoch = int(loaded_state['epoch'])
    logging.info(f"Loaded checkpoint from {checkpoint_path}.")
    print(f"Loaded checkpoint from {checkpoint_path}.")
else:
    logging.warning(f"No checkpoint found at {checkpoint_path}. Starting from scratch.")
    print(f"No checkpoint found at {checkpoint_path}. Starting from scratch.")

model = model.eval()

Loaded checkpoint from run/cosmos_dm_1900_2/checkpoints/checkpoint.pth.


In [5]:
total_time = 0
peak_memory = 0
# Synchronization for timing
starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
count = 0
n_samples = 1
Nside = 256

for sample_no in range(args.start, args.end):
    print(f'Sampling {sample_no}')
    # z0_path = os.path.join(data_root, get_filepath(sample_no, input_type))
    z0_path = f'../Datasets/Quijote_processed/Z0_{Nside}/{sample_no}.npy'

    # === Load z=0 and add Gaussian noise ===
    # N = config.data.image_size
    z0 = np.load(z0_path).reshape(Nside, Nside, Nside)
    noise_sigma = config.data.noise_sigma
    z0_noisy = z0 + noise_sigma * np.random.normal(size=z0.shape)
    z0_noisy = z0_noisy[np.newaxis, ...]  # shape: (1, 128, 128, 128)

    enable_tqdm = not args.disable_tqdm

    sigma_time = get_sigma_time(config.model.sigma_min, config.model.sigma_max)
    sample_time = get_sample_time(config.model.sampling_eps, config.model.T)

    input_data = torch.from_numpy(np.float32(z0_noisy)).to(DEVICE)
    input_data = torch.unsqueeze(input_data, dim=1)

    torch.cuda.reset_peak_memory_stats(DEVICE)
    
    # Start Timing
    starter.record()

    # %%
    def one_step(x, t):
        t_vec = torch.ones(shape[0], device=DEVICE) * t
        model_output = model(torch.cat([x, input_data], dim=1), t_vec)
        x, x_mean = sde.update_fn(x, t_vec, model_output=model_output)
        return x, x_mean

    input_data = torch.tile(input_data, dims=(1, 1, 1, 1, 1))
    shape = (1, 1, Nside, Nside, Nside)

    for j in tqdm(
        range(n_samples)
    ):
        with torch.no_grad(), ema.average_parameters():
            x = sde.prior_sampling(shape).to(DEVICE)
            timesteps = sde.timesteps.to(DEVICE)

            for i in tqdm(range(sde.N)):
                t = timesteps[i]
                x, x_mean = one_step(x, t)

    count += 1
    # End Timing
    ender.record()
    torch.cuda.synchronize()
    
    # Metrics calculation
    total_time += starter.elapsed_time(ender) / 1000 # convert ms to seconds
    peak_memory = max(peak_memory, torch.cuda.max_memory_allocated(DEVICE) / (1024**2)) # MB

# --- Final Report ---
avg_time = total_time / (n_samples*count)

print(f"\n{'='*30}")
print(f"Benckmark Results (Avg per sample):")
print(f"Time: {avg_time:.4f} seconds")
print(f"Peak Memory: {peak_memory:.2f} MB")

Sampling 1999


100%|██████████| 1/1 [31:29<00:00, 1889.49s/it]


Benckmark Results (Avg per sample):
Time: 1889.4927 seconds
Peak Memory: 31192.32 MB


# Quijote

## 32

Time: 10.7712 seconds
Peak Memory: 223.43 MB

## 64

Time: 41.0360 seconds
Peak Memory: 810.72 MB

## 128

Time: 248.3850 seconds
Peak Memory: 4032.32 MB

## 256 

Time: 1889.4927 seconds
Peak Memory: 31192.32 MB